In [3]:
import argparse
import os
import pickle
import sys
import typing

import pandas as pd
import torch
from Bio import SeqIO
from typing import List, Union, Optional, Callable, Sequence
from transformers import (
    EsmForMaskedLM, 
    PretrainedConfig, 
    EsmTokenizer, 
    DataCollatorForLanguageModeling, 
    Trainer
)

from tokenizers import Tokenizer
import torch
import torch.nn.functional as F

import einops
import yaml
import sys
import json
import functools
import os

import numpy as np
from huggingface_hub import hf_hub_download
from datasets import Dataset, load_dataset
import math

from matplotlib import pyplot as plt

In [4]:
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookedRootModule,
    HookPoint,
)

# Hooking utilities
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    FactoredMatrix,
    ActivationCache,
)

sys.path.append("../../scripts")
from compute_node_embeddings import load_sequences, get_protein_sequence
from interp_utils import get_hooked_state_dict, get_hooked_esm_config, get_logits_hooked_esm

## Loading ESM and hooked ESM

In [6]:
device = "cuda"
CONTEXT_LEN = 1000

In [7]:
# ESM-2 config
with open("./esm_config150M.json", "r") as file:
    esm_config = json.load(file)
model_name = esm_config["_name_or_path"]
model_name = model_name[model_name.find("facebook"):]
esm_config["token_dropout"] = False
esm_config["model_name"] = model_name
esm_config = PretrainedConfig(**esm_config)

# ESM-2 tokenizezr config
REPO_ID = esm_config.model_name
special_tokens_map_file = "special_tokens_map.json"
tokenizer_config = {}
tokenizer_config["vocab_file"] = hf_hub_download(repo_id=REPO_ID, filename="vocab.txt")
tokenizer_config["model_max_length"] = CONTEXT_LEN
with open(hf_hub_download(repo_id=REPO_ID, filename=special_tokens_map_file), "r") as f:
    tokenizer_config = {**tokenizer_config, **(json.load(f))}

# hooked ESM-2 config
hooked_esm_config = get_hooked_esm_config(esm_config, context_len=CONTEXT_LEN)

In [8]:
# ESM-2
model_name = esm_config.model_name
print(model_name)

tokenizer = tokenizer = EsmTokenizer(**tokenizer_config)
reg_esm = EsmForMaskedLM(esm_config).to(device)

reg_esm_state_dict = torch.load(hf_hub_download(repo_id=REPO_ID, filename="pytorch_model.bin"))
# removing fixed position embedding
del reg_esm_state_dict["esm.embeddings.position_embeddings.weight"]
del reg_esm_state_dict["esm.embeddings.position_ids"]

print(reg_esm.load_state_dict(reg_esm_state_dict))

facebook/esm2_t30_150M_UR50D
<All keys matched successfully>


In [9]:
# hooked ESM-2
hooked_esm = HookedTransformer(hooked_esm_config)
print(hooked_esm.load_state_dict(get_hooked_state_dict(reg_esm.state_dict(), hooked_esm_config)))

# helper function to get logits from hooked ESM-2 head
apply_esm_lm_head = functools.partial(get_logits_hooked_esm, ESM2_lm_head=reg_esm.lm_head)

<All keys matched successfully>


## Data Loading

In [10]:
# data loading
with open("../../config/pathogen_config.yaml", "r") as config_file:
    config = yaml.safe_load(config_file)
    
pathogens = list(config["pathogens"].keys())
pathogen_idx = 7
print(pathogens[pathogen_idx])
fasta_file = f"../../data/pathogen/{pathogens[pathogen_idx]}/alignment.fasta"
data = load_sequences(fasta_file)
sequence_names, sequences = list(zip(*list(data.items())))

influenza_h1n1pdm_ha


In [11]:
N = 0
test_seq = np.unique(sequences)[50]
# test_seq = np.unique(sequences)[1000]
print(test_seq)
tokenized_seq = tokenizer(test_seq, return_tensors="pt", padding=False).input_ids[:,:hooked_esm_config.n_ctx].to(device)
print(tokenized_seq.shape)

DTLCIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDKHDGKLCKLRGVAPLHLGQCNIAGWILGNPECESLSTASSWSYIVETPSSDNGTCYPGDFIDYEELREQLSSVSSFERFEIFPKASSWPNHDSDNGVTAACPHAGAKSFYKNLIWLVKKGKSYPKISKSYINDQGKEVLVLWGIHHPSTITDQESLYQNADAYVFVGSSRYSKKFKPEIAIRPKVRDRAGRMNYYWTLVEPGDKITFEATGNLVAPRYAFAMKKDAGSGIIISDTPVHDCNTTCQTPKGAINTSLPFQNIHPITIGKCPKYVRSTKLRLATGLRNIPSIQSR
torch.Size([1, 329])


In [12]:
# run respective models
with torch.no_grad():
    outputs = reg_esm.forward(tokenized_seq, output_hidden_states=True)
    reg_esm_hs = outputs.hidden_states

hooked_toks, cache = hooked_esm.run_with_cache(tokenized_seq)

In [13]:
for l in range(esm_config.num_hidden_layers + 2):
    # 0th layer is embedding layer
    if l < esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"blocks.{l}.hook_resid_pre"]
    
    # final layer
    elif l == esm_config.num_hidden_layers:
        actual_output = reg_esm_hs[l]
        hooked_output = cache[f"ln_final.hook_normalized"]

    # ESM LM head:
    else:
        actual_output = outputs.logits
        with torch.no_grad():
            hooked_output = apply_esm_lm_head(hooked_toks)
    
    is_close = torch.all(torch.isclose(actual_output, hooked_output, rtol=0.1, atol=1e-8)).item()
    print(f"layer {l}: {is_close}")

layer 0: True
layer 1: True
layer 2: False
layer 3: False
layer 4: False
layer 5: False
layer 6: False
layer 7: False
layer 8: False
layer 9: False
layer 10: False
layer 11: False
layer 12: False
layer 13: False
layer 14: False
layer 15: False
layer 16: False
layer 17: False
layer 18: False
layer 19: False
layer 20: False
layer 21: False
layer 22: False
layer 23: False
layer 24: False
layer 25: False
layer 26: False
layer 27: False
layer 28: False
layer 29: False
layer 30: False
layer 31: True
